# 01 Screening Examples

This notebook demonstrates how to fine-tune an encoder model for binary screening.

Learning goals:
- Create a toy screening dataset
- Split data into train and test sets
- Tokenize text with DistilBERT tokenizer
- Fine-tune and evaluate a classifier
- Save and reload the model for prediction

⚠ Don't forget to select the CESAB 2026 virtual environment kernel on the upper-right hand side.

In [1]:
from pathlib import Path
import gc
import os

import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
    pipeline,
)

Create example data with the following columns:

* id - a unique row identifier
* text - example text (a sentence) which is about biodiversity or some other topic
* environment_true - a human label of whether the text is about the environment or not (example screening decision)

In [2]:
df = pd.DataFrame({
    "id": list(range(1, 21)),
    "text": [
        "Biodiversity is crucial for healthy ecosystems",
        "The weather today is sunny and warm",
        "Deforestation reduces species diversity",
        "I love playing soccer with friends",
        "Conservation protects endangered species",
        "The stock market rose by 2 percent",
        "Pollinators are essential for crop production",
        "I enjoy listening to jazz music",
        "Marine biodiversity supports fisheries",
        "The new smartphone was released today",
        "Wetlands store carbon and prevent flooding",
        "I baked a cake yesterday",
        "Habitat loss threatens many animals",
        "She went shopping at the mall",
        "Genetic diversity strengthens ecosystems",
        "He bought a new car last week",
        "Ecosystem services benefit human health",
        "The movie was exciting and fun",
        "Forests provide habitat and clean air",
        "I am learning to play guitar"
    ],
    "environment_true": [1, 0] * 10
})

df.head()

,id,text,environment_true
0,1,Biodiversity is crucial for healthy ecosystems,1
1,2,The weather today is sunny and warm,0
2,3,Deforestation reduces species diversity,1
3,4,I love playing soccer with friends,0
4,5,Conservation protects endangered species,1


For simplicity we will just split the data into one training and test set, but be aware that more often k-fold cross validation is used (the data are split into K number of splits, and then each split is used alternatively as the testing set while the others are the training) 

In [3]:
# define training data frames as 70%, testing as 30%
train_df = df.sample(frac=0.7, random_state=42)
test_df = df.drop(train_df.index)

# convert each data frame to a HuggingFace Dataset 
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

len(train_ds), len(test_ds)

(14, 6)

In [4]:
## Define metrics to evaluate model performance

metric_accuracy = evaluate.load("accuracy")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": metric_accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": metric_precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": metric_recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": metric_f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

In [5]:
## define classification model and tokenizer

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0: "not environment", 1: "environment"},
    label2id={"not environment": 0, "environment": 1},
)
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
## tokenize - convert text to integers

# Tokenise the training and testing datasets
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

# Rename labels 
train_ds = train_ds.rename_column("environment_true", "labels")
test_ds = test_ds.rename_column("environment_true", "labels")

# Ensure columns are the correct format 
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

## Calculate attention weights & train the classification layer 

Note that batch size, number of epochs, weight decay, etc are just a few of the hyperparameters that you can optimize (others include loss functions). If you plan on using this approach yourself, you should familiarize yourself with how different combinations of hyperparameters can be looped through to select for the best model configuration using nested cross validation.

But here we will just select arbitrary hyperparameter values as an example. 

In [7]:
from transformers.utils.notebook import NotebookProgressCallback

os.environ["TENSORBOARD_LOGGING_DIR"] = "./outputs/logs"

training_args = TrainingArguments(
    output_dir="./outputs/checkpoints",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to="none",
    disable_tqdm=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Work around a notebook callback bug in some Transformers versions.
trainer.remove_callback(NotebookProgressCallback)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

c:\Users\Devi\R-working-folder\CESAB_AI_lit_review_spring_2026\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.5021', 'eval_accuracy': '1', 'eval_precision': '1', 'eval_recall': '1', 'eval_f1': '1', 'eval_runtime': '0.1077', 'eval_samples_per_second': '55.71', 'eval_steps_per_second': '18.57', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Devi\R-working-folder\CESAB_AI_lit_review_spring_2026\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.4085', 'eval_accuracy': '1', 'eval_precision': '1', 'eval_recall': '1', 'eval_f1': '1', 'eval_runtime': '0.0838', 'eval_samples_per_second': '71.6', 'eval_steps_per_second': '23.87', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '3.595', 'train_samples_per_second': '7.789', 'train_steps_per_second': '2.225', 'train_loss': '0.5556', 'epoch': '2'}
{'eval_loss': '0.4085', 'eval_accuracy': '1', 'eval_precision': '1', 'eval_recall': '1', 'eval_f1': '1', 'eval_runtime': '0.0898', 'eval_samples_per_second': '66.81', 'eval_steps_per_second': '22.27', 'epoch': '2'}
{'eval_loss': 0.40850770473480225, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0, 'eval_runtime': 0.0898, 'eval_samples_per_second': 66.814, 'eval_steps_per_second': 22.271, 'epoch': 2.0}


c:\Users\Devi\R-working-folder\CESAB_AI_lit_review_spring_2026\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [8]:
## Save fine-tuned model

from pathlib import Path

if "trainer" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Run the training cell first so `trainer` and `tokenizer` are available."
    )

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name.lower() == "analyses" else cwd
save_path = repo_root / "data" / "derived-data" / "finetuned_distilbert_biodiv"
save_path.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(save_path))
tokenizer.save_pretrained(str(save_path))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('c:\\Users\\Devi\\R-working-folder\\CESAB_AI_lit_review_spring_2026\\data\\derived-data\\finetuned_distilbert_biodiv\\tokenizer_config.json',
 'c:\\Users\\Devi\\R-working-folder\\CESAB_AI_lit_review_spring_2026\\data\\derived-data\\finetuned_distilbert_biodiv\\tokenizer.json')

Then, as an example of how you would get predictions for unseen text, I'll load the model and print the predictions for one example senetence:

In [9]:
# Load the saved model
nlp = pipeline("text-classification", model=str(save_path), tokenizer=str(save_path))

# Get predictions from an example text
example_text = "Conservation protects endangered species"
print(f"Saved model to: {save_path.resolve()}")
print(nlp(example_text))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Saved model to: C:\Users\Devi\R-working-folder\CESAB_AI_lit_review_spring_2026\data\derived-data\finetuned_distilbert_biodiv
[{'label': 'environment', 'score': 0.6678770780563354}]


In [10]:
del nlp
del tokenizer
del model
gc.collect()

2865


Here we use the model for screening, but you can easily adapt this code for multiple labels (e.g. for coding) by changing the `num_labels` parameter when loading the model.

Here's [another great tutorial for text classification from Hugging Face](https://huggingface.co/docs/transformers/en/tasks/sequence_classification)


FYI:
In real life, we don't usually have an even number of examples of inclusions and exclusions. Often the number of inclusions << exclusions. In this scenario, you want to make sure that you have an adequate number of inclusions in your training data (which can be augmented using keyword searching, active learning, etc). If you use this approach, make sure that only randomly sampled documents are used in the test set.
